#
# Geometric Characterization of the Decision Manifold
#

This notebook reproduces the analyses from **Figure 3: The deliberation-commitment pattern of the decision manifold** of the paper:  

**"The geometry of the neural state space of decisions", Monsalve-Mercado et al. (2025)**  

[https://doi.org/10.1101/2025.01.24.634806](https://doi.org/10.1101/2025.01.24.634806)

---

## Overview

We analyze and visualize the geometric structure of single-trial neural trajectories during perceptual decision-making, as recorded from the lateral intraparietal (LIP) cortex. This notebook focuses on understanding the **decision manifold**—a low-dimensional surface in neural state space—using tools from differential geometry and trial-aligned dimensionality reduction.

---

## What’s Inside

- **Arc-length parameterization** of neural trajectories from dots onset to saccade.
- **PCA** to define a low-dimensional embedding of trial-aligned neural activity.
- **Curvature, speed, and tortuosity** measurements along single-trial and average trajectories.
- Construction of **average manifolds** aligned by **reaction time (RT)**, revealing a smooth, butterfly-shaped decision surface.
- **Tangent space analysis** to extract three components of neural dynamics:
  - **Resolution direction**: progression of the decision process.
  - **Uncertainty direction**: variability in timing along the manifold.
  - **Off-manifold direction**: fluctuations orthogonal to the surface.

---

## Key Insights from the Paper

- The decision manifold is a **two-dimensional, curved surface** shaped by the time-varying dynamics of decision formation.
- Trajectories for left and right choices unfold in **mirror-symmetric branches**, bending away as evidence accumulates.
- **Neural variability is structured**: most fluctuations lie within the tangent space, aligned with either decision progression or timing uncertainty.
- The geometry reveals a **computational separation of deliberation and commitment**, with geometric measures qualitatively transitioning between stages.

---

## Requirements

- Processed data from Zenodo: [doi:10.5281/zenodo.15093134](https://doi.org/10.5281/zenodo.15093134)
- Helper scripts from `src/` for:
  - Data loading (`io_utils.py`)
  - Geometry computations (`geometry.py`)
  - Plotting utilities (`plot_utils.py`)

---



##
## Load the data

In [ ]:
import os
import sys

# Set the root directory

# Automatically find the project root (directory containing 'src' or 'data')
def find_project_root(marker_dirs=("src", "data","notebooks")):
    path = os.getcwd()
    while path != "/" and not all(os.path.exists(os.path.join(path, d)) for d in marker_dirs):
        path = os.path.dirname(path)
    return path

PROJECT_ROOT = find_project_root()

# Set up Python import path and working directory
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)





# Download/load data
# Also sets the working session

from io_utils import download_session
from src.io_utils import load_dataframe_with_metadata

session = "S6"
path = download_session(session)
df = load_dataframe_with_metadata(session)

##
## PCA Dimensionality Reduction

First some preprocessing: 

Align trials at dot-motion onset to the trial-average population vector. This helps reducing the effect of activity drift in non-coding direction.

Crop time 200ms after dot-motion onset -- the approximate time when evidence integration starts -- until 60ms before saccade completed -- to avoid some motor action preparation. 

Smooth activity with a gaussian kernel (sigma=50ms).

In [ ]:
from scipy.ndimage import gaussian_filter1d
import numpy as np

# Crop time and align trials LFADS
Cut = 20     # time steps (10ms bins) after dotsOn to trim start
CutEnd = -6  # time steps before saccadeDetected to trim end
sigma = 5    # smoothing kernel width

# Smooth LFADS trajectories
for trial in df.index:
    df.at[trial, 'LFADS'] = gaussian_filter1d(df.at[trial, 'LFADS'], sigma=sigma, axis=1)

# Compute reference starting points for alignment
start_vectors = np.stack([df.at[trial, 'LFADS'][:, 0] for trial in df.index], axis=1)
mean_start = np.mean(start_vectors, axis=1, keepdims=True)

# Align and cut LFADS + store back
for trial in df.index:
    traj = df.at[trial, 'LFADS']
    aligned = traj[:, Cut:CutEnd] - traj[:, [0]] + mean_start
    df.at[trial, 'LFADS'] = aligned


We reduce the dataset to the first 10 principal components. This makes a new column 'LFADS-PCA'.

In [ ]:
from src.geometry import get_pca

principal = get_pca(df, column='LFADS', cells=None, components=10, plot=True)


##
## Arc-length reparametrization
Compute each trial's arc-length in Hz (`Arc-Length-Proper`) and a normalized equivalent (`Arc-Length`). 

Reparametrize time, speed, the trajectories, curvature, and tortuosity in arc-length. The function adds the new columns to the dataframe in place, appending the suffix `-Arc` to the name.

In [ ]:
from src.geometry import compute_geometry_measures

compute_geometry_measures(df, column='LFADS-PCA', timeres=10, arc_res=101)
df.keys()[-12:]

##
## Representation in the manifold 2D local coordinates

We can now compare trials at a given stage of the decision via their arc-length value. Since choice-submanifolds are locally organized by reaction time, we can leverage this continuous behavioural axis to find a locally averaged representation of any measure on the manifold's local coordinates (arc-length and reaction-time).

For example, we can compute this local average for the speed and tortuosity for the contralateral submanifold (`choice=0`). The function outputs an array of shape `arc-length points x reaction-time points` (`101 x 101` here).

In [ ]:
from src.geometry import local_average

speed_0 , nba_0 = local_average(df[df['choice']==0], column='Speed-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)
tortuosity_0 , nba_0 = local_average(df[df['choice']==0], column='Tortuosity-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)




In [ ]:

from src.plot_utils import plot_local_average

plot_local_average(data=speed_0, behavioral_axis=nba_0, var_name='Speed (Hz/ms)').show()
plot_local_average(data=tortuosity_0, behavioral_axis=nba_0, var_name='Tortuosity').show()


Similarly we can compute the local average representation of the neural population firing rate activity. 

The function outputs an array of shape `PCA components x arc-length points x reaction-time points`.

ALternatively, one can compute it directly on each neuron's activity (slower), but must first compute `LFADS-Arc` (not done here).

In [ ]:
from src.geometry import local_average

# choice 0 and 1 correspond to contralateral and ipsilateral choice respectively
pop_0 , nba_0 = local_average(df[df['choice']==0], column='LFADS-PCA-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)
pop_1 , nba_1 = local_average(df[df['choice']==1], column='LFADS-PCA-Arc', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)


In [ ]:
from src.plot_utils import plot_local_average_pop

fig = plot_local_average_pop(
    manifolds=[pop_0, pop_1],
    behavioral_axes=[nba_0, nba_1],
    color_ranges=[("gray blue", "neon blue"), ("gray pink", "neon pink")],
    names=['Contralateral Choice', 'Ipsilateral Choice']
)
fig.show()
#fig.write_html('AverageManifold.html')

#

The local average estimates the deterministic manifold that would result from the underlying dynamical system running under "noise free" input.

The euclidean distance between equivalent points in the two choice submanifolds --equivalent in the normalized local coordinate system (both axes running from 0 to 1), shows that choices separate later for longer reaction times.

The curvature of lines or constant reaction-time value shows that the manifold curves in non-trivial directions in neural space.

In [ ]:
from src.geometry import K

distance_0 = np.linalg.norm(pop_0-pop_1,axis=0)
curvature_0 = 180/np.pi*np.stack([K(pop_0[:, :, i],w=1) for i in range(pop_0.shape[2])], axis=1)


from src.plot_utils import plot_local_average

plot_local_average(data=distance_0, behavioral_axis=nba_0/np.max(nba_0), var_name='Distance (Hz)',title='Local Distance between Choice-Submanifolds',ba_title='Reaction Time (normalized)').show()
plot_local_average(data=curvature_0, behavioral_axis=nba_0, var_name='Curvature (deg/Hz)',title='Submanifold Curvature in the Resolution Direction').show()


The alignment of tangent vectors at equivalent points between the two choice-submanifolds is consistent with slower trials separating later during decision-making

In [ ]:
from src.geometry import tangent_space
from scipy.ndimage import gaussian_filter as gf2

# compute unit tangent vectors
t0 = tangent_space(pop_0, axis=1)
t1 = tangent_space(pop_1, axis=1)

dot = np.einsum('ntm,ntm->tm', t0, t1)
norms = np.linalg.norm(t0, axis=0) * np.linalg.norm(t1, axis=0)
norms[norms == 0] = np.nan 

# compute cosine similarity between unit tangent vectors
cosine_similarity = gf2(dot / norms, sigma=5)


from src.plot_utils import plot_local_average

plot_local_average(data=cosine_similarity, behavioral_axis=nba_0/np.max(nba_0), var_name='Cosine Similarity',title='Local Cosine Similarity between Choice-Submanifolds',ba_title='Reaction Time (normalized)').show()


##
## Tangent Space Decomposition

Computing the tangent spaces of the average manifold allows us to decompose single-trial unit tangent vectors into meaningful geometric components:

- **Off-manifold component**: orthogonal to the local tangent plane.
- **On-manifold components** (within the 2D tangent plane):
  - **Resolution direction**: aligned with the direction of increasing arc-length.
  - **Uncertainty direction**: orthogonal to the resolution direction within the plane, roughly aligned with decreasing reaction time.

This decomposition defines a local coordinate system for interpreting trial-by-trial variability in terms of behaviorally meaningful directions.


In [ ]:
from src.geometry import tangent_space
from src.geometry import compute_unit_orthogonal_vectors
from src.geometry import get_components

# compute reference tangent spaces
tangent_0 = tangent_space(pop_0, axis=1) # along arc-length
transverse_0 = tangent_space(pop_0, axis=2) # along reaction-time
transverse_0 = compute_unit_orthogonal_vectors(tangent_0, transverse_0, axis=0)


# Compute squared components of projections of the single-trial unit tangent vectors into the resolution and uncertainty directions of the average manifold
get_components(df, tangent_ref=tangent_0, conditions=[df['choice']==0], new_behavioral_axis=nba_0, name='Resolution')
get_components(df, tangent_ref=transverse_0, conditions=[df['choice']==0], new_behavioral_axis=nba_0, name='Uncertainty')

df['OffManifoldSquareProjection'] = 1 - df['ResolutionSquareProjection'] - df['UncertaintySquareProjection']


resolution_0 , nba_0 = local_average(df[df['choice']==0], column='ResolutionSquareProjection', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)
uncertainty_0 , nba_0 = local_average(df[df['choice']==0], column='UncertaintySquareProjection', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)
offmanifold_0 , nba_0 = local_average(df[df['choice']==0], column='OffManifoldSquareProjection', behavioral_axis='RT', frac=0.5, ba_res=101, method='lowess', remove_outliers=0.01)

In [ ]:
from src.plot_utils import plot_local_average

plot_local_average(data=resolution_0, behavioral_axis=nba_0, var_name='Component',title='Squared Projection in the Resolution Direction').show()
plot_local_average(data=uncertainty_0, behavioral_axis=nba_0, var_name='Component',title='Squared Projection in the Uncertainty Direction').show()
plot_local_average(data=offmanifold_0, behavioral_axis=nba_0, var_name='Component',title='Squared Projection Off-Manifold').show()
